In [6]:
# Main code generated by Qwen3.5 Plus #

import serial
import time
import sys

# ================= 配置区域 =================
SERIAL_PORT = 'COM5'      # Windows: 'COM3', Linux/Mac: '/dev/ttyUSB0'
BAUD_RATE = 9600
TIMEOUT = 1.0             # 读取超时时间 (秒)
HEADER_BYTE = b'\x2C'     # 包头字节 0x2C
PACKET_LENGTH = 12        # 总包长度
DEBUG = 1
# ===========================================

def parse_temperature(raw_value):
    """
    解析温度值，处理负数情况
    文档规则：16位无符号数，最高位为1表示零下。
    计算：0xFFFF - 当前数据 = 绝对值，然后加负号。
    单位：0.1°C
    """
    if raw_value & 0x8000:
        abs_value = 0xFFFF - raw_value
        return -abs_value * 0.1
    else:
        return raw_value * 0.1

def parse_voc_data(data_bytes):
    """
    解析12字节数据包
    """
    if len(data_bytes) != PACKET_LENGTH:
        return None

    # 1. 验证包头
    if data_bytes[0] != 0x2C:
        print(f"[错误] 包头校验失败: 期望 0x2C, 实际 {hex(data_bytes[0])}")
        return None

    # 2. 按文档公式 Data[n]*(2^8) + Data[n+1] 解析
    multiplier = 256
    
    voc = data_bytes[1] * multiplier + data_bytes[2]
    formaldehyde = data_bytes[3] * multiplier + data_bytes[4]
    eco2 = data_bytes[5] * multiplier + data_bytes[6]
    
    temp_raw = data_bytes[7] * multiplier + data_bytes[8]
    temperature = parse_temperature(temp_raw)
    
    humidity_raw = data_bytes[9] * multiplier + data_bytes[10]
    humidity = humidity_raw * 0.1

    # 3. 校验和验证 (前11字节累加，取反+1)
    calculated_checksum = (~sum(data_bytes[:11]) + 1) & 0xFF
    received_checksum = data_bytes[11]
    
    if calculated_checksum != received_checksum:
        print(f"[警告] 校验和失败! 计算: {hex(calculated_checksum)}, 接收: {hex(received_checksum)}")
        # 可以选择是否返回数据，这里返回但标记校验失败
        valid = False
    else:
        valid = True

    return {
        "voc": voc,
        "formaldehyde": formaldehyde,
        "eco2": eco2,
        "temperature": temperature,
        "humidity": humidity,
        "valid": valid,
        "raw_hex": data_bytes.hex()
    }

def main():
    try:
        # 打开串口，必须设置 timeout，否则 read_until 会无限阻塞
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=TIMEOUT)
        print(f"已打开串口: {SERIAL_PORT} @ {BAUD_RATE}")
        print("等待数据包 (按 Ctrl+C 退出)...")
        
        while True:
            # 策略：使用 read_until 寻找包头 0x2C
            # read_until 会读取直到遇到 HEADER_BYTE 或者 超时
            # 返回值包含 HEADER_BYTE
            header_chunk = ser.read_until(HEADER_BYTE)
            
            if not header_chunk:
                # 超时未读到任何数据或未到包头
                continue
            
            # header_chunk 的最后一个字节肯定是 0x2C (除非是超时返回的空或部分数据)
            if header_chunk[-1:] != HEADER_BYTE:
                # 这种情况通常是超时了，读到的数据里没有 0x2C
                continue
            
            # 现在我们有了包头，还需要读取剩余的 11 个字节
            # 注意：header_chunk 可能包含包头之前的垃圾数据，我们只取最后一个字节作为头
            # 如果 header_chunk 长度 > 1，说明前面有垃圾数据，已被 read_until 过滤掉，只保留含头的部分
            
            remaining_length = PACKET_LENGTH - 1
            remaining_data = ser.read(remaining_length)
            
            if len(remaining_data) < remaining_length:
                print(f"[警告] 数据包不完整，只读到 {len(remaining_data)}/{remaining_length} 个后续字节")
                continue
            
            # 拼接完整包：包头 (1字节) + 后续数据 (11字节)
            full_packet = HEADER_BYTE + remaining_data
            
            # 解析
            result = parse_voc_data(full_packet)
            
            if result:
                status = "[OK]" if result["valid"] else "[CHKSUM ERR]"
                print(f"{status} VOC:{result['voc']} | HCHO:{result['formaldehyde']} | eCO2:{result['eco2']} | T:{result['temperature']:.1f}°C | H:{result['humidity']:.1f}%", end="")
                if DEBUG: print(f" | Hex:{result['raw_hex']}", end="")
                print()

    except serial.SerialException as e:
        print(f"\n串口错误: {e}")
    except KeyboardInterrupt:
        print("\n用户中断，正在关闭...")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
            print("串口已关闭。")

if __name__ == "__main__":
    # 确保安装了 pyserial
    # pip install pyserial
    main()

已打开串口: COM5 @ 9600
等待数据包 (按 Ctrl+C 退出)...
[OK] VOC:0 | HCHO:0 | eCO2:350 | T:21.6°C | H:52.1% | Hex:2c00000000015e00d8020992
[OK] VOC:4 | HCHO:0 | eCO2:358 | T:21.6°C | H:52.1% | Hex:2c00040000016600d8020986
[OK] VOC:6 | HCHO:0 | eCO2:362 | T:21.6°C | H:52.1% | Hex:2c00060000016a00d8020980
[OK] VOC:4 | HCHO:0 | eCO2:358 | T:21.6°C | H:52.1% | Hex:2c00040000016600d8020986
[OK] VOC:3 | HCHO:0 | eCO2:356 | T:21.6°C | H:52.1% | Hex:2c00030000016400d8020989
[OK] VOC:3 | HCHO:0 | eCO2:356 | T:21.6°C | H:52.2% | Hex:2c00030000016400d8020a88
[OK] VOC:5 | HCHO:0 | eCO2:360 | T:21.6°C | H:52.3% | Hex:2c00050000016800d8020b81
[OK] VOC:376 | HCHO:60 | eCO2:1102 | T:21.9°C | H:55.8% | Hex:2c0178003c044e00db022ec2
[OK] VOC:625 | HCHO:102 | eCO2:1600 | T:22.9°C | H:61.2% | Hex:2c02710066064000e502646a
[OK] VOC:757 | HCHO:124 | eCO2:1864 | T:23.1°C | H:65.1% | Hex:2c02f5007c074800e7028b9e
[OK] VOC:854 | HCHO:140 | eCO2:2058 | T:23.6°C | H:68.9% | Hex:2c0356008c080a00ec02b13e
[OK] VOC:740 | HCHO:121 | 

- VOC (Volatile Organic Compounds)：挥发性有机化合物。Total VOC包括苯、甲苯、甲醛等。 0~4000ug 分辨率 1ug 精度 ±15%
- HCHO (Formaldehyde)：甲醛，0-2000ug (±15%)
- eC02（等效值） 400-5000ppm(±25%) 
- 温度量程：-40-125℃（±0.3℃）
- 湿度量程：0-100%RH（±2%RH） 